In [5]:
from matplotlib import pyplot as plt
import numpy as np

import torch
from sklearn.metrics import classification_report

from tqdm import tqdm

import time

from PieceDetection import PieceDetection

from Dataset.DataSetLoaders import ChessDataset

In [2]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

In [3]:
pc_cnn = PieceDetection.PieceDetector("cnn")
pc_yolo = PieceDetection.PieceDetector("yolo")

/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        interm_time = time.perf_counter()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

100%|██████████| 389/389 [01:39<00:00,  3.90it/s]

Acc: 0.98 | Errors: 1.4087 | Avg Pre-Processing Time: 208.444ms | Avg Processing Time: 28.149ms


In [5]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_yolo.preprocess()
        interm_time = time.perf_counter()
        preds = pc_yolo.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time
        
        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass


avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

100%|██████████| 389/389 [00:19<00:00, 20.34it/s]

Acc: 0.99 | Errors: 0.5090 | Avg Pre-Processing Time: 0.163ms | Avg Processing Time: 17.204ms


In [4]:
train_ds, valid_ds, test_ds = ChessDataset.ChessDataset.train_valid_test_split(ds, sizes=(.8,.1,.1), random_state=42)

In [6]:
accs = []
actual = []
preds_all = []
for img, label in tqdm(test_ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        pc_cnn.preprocess()
        preds = pc_cnn.predict()

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.argmax(dim=1).reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all)
print(rep)

100%|██████████| 40/40 [00:10<00:00,  3.71it/s]

Acc: 0.95 | Errors: 3.0000
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       189
           1       0.59      0.36      0.44        28
           2       0.36      0.75      0.48        20
           3       0.85      0.71      0.78        49
           4       0.72      0.72      0.72        29
           5       0.86      0.93      0.89        40
           6       0.98      0.95      0.97       196
           7       0.58      0.69      0.63        16
           8       0.43      0.65      0.52        20
           9       0.76      0.60      0.67        53
          10       0.86      0.70      0.78        27
          11       0.81      0.88      0.84        40
          12       1.00      1.00      1.00      1853

    accuracy                           0.95      2560
   macro avg       0.75      0.76      0.74      2560
weighted avg       0.96      0.95      0.95      2560



In [8]:
accs = []
actual = []
preds_all = []
for img, label in tqdm(test_ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        pc_yolo.preprocess()
        preds = pc_yolo.predict()

        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all)
print(rep)

100%|██████████| 40/40 [00:02<00:00, 17.49it/s]


Acc: 0.99 | Errors: 0.5250
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       189
           1       0.93      1.00      0.97        28
           2       1.00      0.95      0.97        20
           3       0.96      0.94      0.95        49
           4       0.97      0.97      0.97        29
           5       0.97      0.95      0.96        40
           6       1.00      0.98      0.99       196
           7       1.00      1.00      1.00        16
           8       0.91      1.00      0.95        20
           9       1.00      0.98      0.99        53
          10       1.00      1.00      1.00        27
          11       1.00      0.97      0.99        40
          12       0.99      1.00      1.00      1853

    accuracy                           0.99      2560
   macro avg       0.98      0.98      0.98      2560
weighted avg       0.99      0.99      0.99      2560

